# Testing the Pydantic classes in execution
Init: Week of 14 Nov

Status: Complete and Debugged. Needs fallback methods + traceability
Solved

In [2]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from typing import List, Optional, Literal, Type
import json

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
# Pydantic classes

class DocumentSummary(BaseModel):
    """
    Single flattened Pydantic model containing all extracted results.
    """

    short_summary_text: str = Field(
        ...,
        description="Condensed version of the document’s main points (≤ 3 sentences)."
    )

    clean_text: str = Field(
        ...,
        description=(
            'Fully cleaned and formatted version of the document: grammar fixes, '
            'structured sections, corrected formatting, preserved technical terms.'
        )
    )

    key_terms: Optional[str] = Field(
        None,
        description="Comma-separated list of key terms extracted from the document."
    )

# System Prompt
SYSTEM_PROMPT = """You are an expert note processing tutor that performs. Follow these rules:

# Rule 1: Writing style and tone

## WRITING STYLE GUIDELINES
Write as a student who:
- rewrites their notes neatly after class,
- organizes content into clear thematic sections,
- uses smooth transitions between topics,
- formats everything consistently,
- explains concepts clearly **only with the information already present**,
- never invents new facts,
- removes distractions and confusion,
- formats code and formulas properly.

## TONE
Tone should be:
- clear
- friendly
- calm
- helpful
- academic but approachable


# Rule 2: Extraction Rules
## Extraction Rules:
1. **Preserve all actual information from the notes**, including formulas, lists, definitions, code, and explanations.
2. **Remove nonsense or noise**: corrupted text, random numbers, stray characters, duplicated broken lines, malformed fragments that do not convey meaning.
3. **Repair broken text**: reconstruct incomplete sentences or code *using only the information present*.
4. **Reorganize the content logically**: merge related pieces, introduce clear section headers, and follow a flow similar to a well-organized lecture summary.
5. **Produce a polished, structured markdown document** similar in clarity and tone to well-written student notes.

### What NOT to do:
- Don't add information not present in the source
- Don't make assumptions about implicit meanings
- Don't include general knowledge about the topic
- Don't create definitions for terms not defined in the text
- Don't standardize or normalize technical content



"""

class DocStorageCleaner:
    """
    Extracts structured, strictly text-grounded summaries from Markdown documents
    into a Pydantic schema (e.g., DocumentSummary).
    No inference, guessing, or external knowledge is permitted.
    """

    def __init__(self, model: str = "gemini-2.5-flash"):
        self.model_name = model
        self.client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        self.system_instruction = SYSTEM_PROMPT

    def _build_prompt(self, markdown_doc: str) -> str:
        """Builds the extraction prompt used for the summarization agent."""
        return f"""
You are an educational summarization agent.
Your task is to extract information ONLY from the given Markdown document, with zero interpretation, speculation, or external knowledge.

# HOW TO FILL THE PYDANTIC MODELS:

## For `clean_text` (CleanText model):
### What to look for:
Your task is to transform the raw document into a **clear, readable, well-structured, student-friendly markdown document**, while staying **100% faithful to the original content**.  
### CLEANING & RESTRUCTURING RULES

#### 1. Improve grammar and clarity
- Fix grammar, punctuation, spacing, and awkward phrasing.
- Convert fragments into readable sentences.
- Keep everything concise but complete.

#### 2. Restructure into clean markdown sections  
Use a clear hierarchy:
- `#` for the main topic  
- `##` for major sections  
- `###` for concepts  
- `####` for details  

Group related ideas even if they were scattered across the notes.

#### 3. Format all code properly
- Clean corrupted characters like `·`, `l'accuracy`, broken parentheses, etc.
- Convert code into proper fenced code blocks with language markers:
  ```python
  ...


## For `short_summary` (ShortSummary model):
- Write 2-3 sentences maximum (≤ 3 sentences is strict requirement)
- First sentence: State the main topic/subject of the document
- Second sentence: Mention 2-4 most important concepts or points covered
- Third sentence (optional): State the purpose or application if explicitly mentioned
- Use only information directly stated in the document
- Do NOT add context, background, or explanations not in the source
- **Output**: Concise text (2-3 sentences) that captures essence without detail

## For `key_terms` (List of KeyTerms models):
- Extract ONLY the key terms defined or introduced in the document.
- Extract the main three keywords, only
Return as a comma-separated string, e.g.: "Term1, Term2, Term3"


DOCUMENT:
---
{markdown_doc}
---
        """.strip()

    def summariser_action(self, markdown_doc: str, pydantic_model) -> DocumentSummary:
        """
        Summarizes the markdown document into the DocumentSummary model.
        """
        prompt = self._build_prompt(markdown_doc)

        config = types.GenerateContentConfig(
            system_instruction=self.system_instruction,
            response_mime_type="application/json",
            response_schema=pydantic_model,
        )

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt,
            config=config,
        )

        result = DocumentSummary.model_validate_json(response.text)
        print("Extraction successful!")
        return result

In [48]:
f = open("file.md", 'r') 
markdown_content = f.read()  # Store the content
f.close()

In [49]:
# API implementation

summarizer = DocStorageCleaner(model="gemini-2.0-flash")

# ---- Run the summarization ----
summary = summarizer.summariser_action(markdown_content, pydantic_model=DocumentSummary)



Extraction successful!


In [50]:
print(summary)

short_summary_text="This document covers colored CNNs, padding techniques, image transformations, and transfer learning. It describes different padding methods like 'valid,' 'same,' reflective, and circular padding, along with their effects on image convolution. The document also discusses image preprocessing steps, data augmentation, and the use of transfer learning to leverage pre-trained models for new tasks." clean_text='# Class 13- Coloured CNNs\n\n- Image classification uses filters to learn patterns and is universal to spatial imagery.\n- Images in corners are not centered to the middle of the filter, so padding adds zeros in corners for better retention.\n- Padding is a parameter used inside the convolutional layer to add layers of zeros to input images.\n- 255 pixels for normal RGBs require normalization.\n- Pooling reduces space by extracting features.\n- The flatten layer converts everything into a 1D vector.\n- Use convolution and dense layers to classify.\n- Key parameters

In [51]:
summary.short_summary_text

"This document covers colored CNNs, padding techniques, image transformations, and transfer learning. It describes different padding methods like 'valid,' 'same,' reflective, and circular padding, along with their effects on image convolution. The document also discusses image preprocessing steps, data augmentation, and the use of transfer learning to leverage pre-trained models for new tasks."

In [52]:
print(summary.clean_text)

# Class 13- Coloured CNNs

- Image classification uses filters to learn patterns and is universal to spatial imagery.
- Images in corners are not centered to the middle of the filter, so padding adds zeros in corners for better retention.
- Padding is a parameter used inside the convolutional layer to add layers of zeros to input images.
- 255 pixels for normal RGBs require normalization.
- Pooling reduces space by extracting features.
- The flatten layer converts everything into a 1D vector.
- Use convolution and dense layers to classify.
- Key parameters: learning rate, batch size, optimizer.

## Padding Techniques in Convolutional Neural Networks (CNN)

Padding is a technique used in CNNs to control how the borders of an image are handled during the convolution operation. There are several approaches to applying padding, each with a specific effect on the model's outputs.

## Types of Padding

### 1. Padding = 'valid'

- Description: No padding (zeros) is added to the edges of the i

In [53]:
summary.key_terms

'Padding, Convolutional Neural Networks, Transfer Learning'